# 市场广度 / TRIN 曲线与检验

读取 `data/` 下的两个 CSV，输出图片到 `charts/`，并重写 `README.md`（仓库首页直接看结果）。

TRIN 用标准 Arms Index：(上涨家数 / 下跌家数) / (上涨成交量 / 下跌成交量)。
参考线：Arms 原始 10 日均线阈值 0.8（超买）/ 1.2（超卖）；博主的单日阈值 0.7（他称"多头主动进攻"）。

## 出图

准备和五张图在同一格里，运行这一格即可全部显示（图也保存到 `charts/`）。

1. 上证 / 市场广度 / TRIN 总览
2. 博主提到的几天：家数占比 vs 成交量占比 vs 成交额
3. 权重 vs 中小盘 TRIN
4. 中信期货与前 20 会员净持仓
5. 按 TRIN 区间分组的后续涨跌

In [ ]:
# ===== 准备：字体、配色、读取数据 =====
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker
from matplotlib import font_manager

import glob, subprocess


def pick_cjk_font():
    """找中文字体；Colab / Linux 没有时自动安装 fonts-noto-cjk。"""
    names = ["Noto Sans CJK SC", "Noto Sans CJK JP", "WenQuanYi Zen Hei", "SimHei", "Microsoft YaHei",
             "PingFang SC", "Hiragino Sans GB", "Heiti SC", "STHeiti", "Songti SC", "Arial Unicode MS"]
    have = {f.name for f in font_manager.fontManager.ttflist}
    for n in names:
        if n in have:
            return n
    files = glob.glob("/usr/share/fonts/**/NotoSansCJK*", recursive=True)
    if not files and os.path.exists("/usr/bin/apt-get"):
        print("未找到中文字体，正在安装 fonts-noto-cjk（约 1 分钟）…")
        subprocess.run("apt-get -qq update && apt-get -qq install -y fonts-noto-cjk", shell=True)
        files = glob.glob("/usr/share/fonts/**/NotoSansCJK*", recursive=True)
    for f in files:
        font_manager.fontManager.addfont(f)
    have = {f.name for f in font_manager.fontManager.ttflist}
    for n in names:
        if n in have:
            return n
    print("⚠ 没有找到中文字体，图中中文可能显示为方框")
    return None


_font = pick_cjk_font()
if _font:
    plt.rcParams["font.sans-serif"] = [_font]
plt.rcParams.update({
    "axes.unicode_minus": False, "figure.dpi": 110, "savefig.dpi": 150,
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
    "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlecolor": "#0b0b0b",
    "legend.frameon": False, "legend.fontsize": 9,
})
BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948")
INK2, MUTED = "#52514e", "#898781"

os.makedirs("charts", exist_ok=True)
assert os.path.exists("data/a_breadth.csv"), "找不到 data/a_breadth.csv：请先运行上面的拉数据部分"
b = pd.read_csv("data/a_breadth.csv", dtype={"trade_date": str})
b["date"] = pd.to_datetime(b.trade_date)
b = b.sort_values("date").reset_index(drop=True)
fut = pd.read_csv("data/cffex_holding.csv", dtype={"trade_date": str}) if os.path.exists("data/cffex_holding.csv") else pd.DataFrame()

KEY = {"20260323": "3/23", "20260721": "7/21", "20260921": "9/21",
       "20260922": "9/22", "20261022": "10/22", "20261023": "10/23"}
key = b[b.trade_date.isin(KEY)].copy()
key["label"] = key.trade_date.map(KEY)
last = b.iloc[-1]
print(f"数据区间 {b.trade_date.iloc[0]} ~ {last.trade_date}，共 {len(b)} 个交易日")


def mark_keys(ax, top=False):
    prev = None
    for i, (_, r) in enumerate(key.iterrows()):
        ax.axvline(r.date, color=MUTED, lw=0.8, ls=":", zorder=0)
        if top:
            near = prev is not None and (r.date - prev).days < 20
            dy = -14 if near and i % 2 else -2
            ax.annotate(r.label, (r.date, 1), xycoords=("data", "axes fraction"),
                        xytext=(2, dy), textcoords="offset points", fontsize=8, color=INK2, va="top")
            prev = r.date


def fmt_x(ax):
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))


def trin_axis(ax):
    ax.set_yscale("log")
    ax.set_yticks([0.3, 0.5, 0.7, 0.8, 1, 1.2, 1.5, 2, 3])
    ax.get_yaxis().set_major_formatter(matplotlib.ticker.FormatStrFormatter("%g"))
    ax.get_yaxis().set_minor_formatter(matplotlib.ticker.NullFormatter())
    ax.axhline(1, color=MUTED, lw=0.8)
    ax.axhline(0.8, color=GREEN, lw=0.8, ls="--")
    ax.axhline(1.2, color=RED, lw=0.8, ls="--")

# ===== 图 1：上证指数 / 市场广度 / TRIN（单日 + 10 日均线）

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True, gridspec_kw={"height_ratios": [1.2, 1, 1.2]})
ax = axes[0]
ax.plot(b.date, b.sh_close, color=BLUE, lw=1.6)
ax.set_title("上证指数")
mark_keys(ax, top=True)
ax = axes[1]
ax.plot(b.date, b.breadth_up * 100, color=MUTED, lw=0.7)
ax.plot(b.date, (b.breadth_up * 100).rolling(10).mean(), color=AQUA, lw=2, label="10 日均线")
ax.set_title("市场广度：上涨家数占比（%）")
ax.set_ylim(0, 100)
ax.legend(loc="upper left")
mark_keys(ax)
ax = axes[2]
ax.plot(b.date, b.trin, color=MUTED, lw=0.7, label="单日 TRIN")
ax.plot(b.date, b.trin_ma10, color=BLUE, lw=2, label="10 日均线")
trin_axis(ax)
ax.set_title("TRIN（对数坐标；虚线 0.8 超买 / 1.2 超卖，Arms 原始阈值）")
ax.legend(loc="upper left", ncol=2)
mark_keys(ax)
fmt_x(ax)
fig.tight_layout()
fig.savefig("charts/1_overview.png")
plt.show()

# ===== 图 2：博主提到的几天——家数占比 vs 成交量占比 vs 成交额
#
# 如果"上涨股爆量"，上涨股的成交量占比应明显高于家数占比。TRIN 只反映这两个占比的相对关系，不反映绝对成交量。

if not key.empty:
    k = key.copy()
    k["cnt_share"] = k.n_up / (k.n_up + k.n_dn) * 100
    k["vol_share"] = k.up_vol / (k.up_vol + k.dn_vol) * 100
    k["amt_share"] = k.up_amt / (k.up_amt + k.dn_amt) * 100
    x = np.arange(len(k))
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1.6, 1]})
    ax = axes[0]
    w = 0.26
    for i, (col, name, c) in enumerate([("cnt_share", "上涨家数占比", MUTED),
                                         ("vol_share", "上涨股成交量占比", BLUE),
                                         ("amt_share", "上涨股成交额占比", ORANGE)]):
        bars = ax.bar(x + (i - 1) * w, k[col], w * 0.92, color=c, label=name)
        ax.bar_label(bars, fmt="%.0f", fontsize=8, color=INK2, padding=2)
    ax.set_xticks(x, [f"{l}\nTRIN {t:.2f}" for l, t in zip(k.label, k.trin)])
    ax.set_ylim(0, 115)
    ax.set_title("上涨股：家数占比 vs 成交量 / 成交额占比（%）")
    ax.legend(loc="upper left", ncol=3)
    ax.grid(axis="x", visible=False)
    ax = axes[1]
    med = b.amt.rolling(60).median()
    bars = ax.bar(x, k.amt, 0.6, color=BLUE)
    ax.bar_label(bars, fmt="%.0f", fontsize=8, color=INK2, padding=2)
    ax.scatter(x, med[k.index], color=ORANGE, marker="_", s=600, lw=2, zorder=3, label="此前 60 日成交额中位数")
    ax.set_xticks(x, k.label)
    ax.set_ylim(0, max(k.amt.max(), med[k.index].max()) * 1.22)
    ax.set_title("当日两市成交额（亿元）")
    ax.legend(loc="upper left")
    ax.grid(axis="x", visible=False)
    fig.tight_layout()
    fig.savefig("charts/2_key_days.png")
    plt.show()

# ===== 图 3：权重（市值前 300）vs 中小盘的 TRIN 10 日均线
#
# 检验"权重放量出货、小票补涨"：如果成立，权重组 TRIN 应明显高于中小盘组。

fig, axes = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True, gridspec_kw={"height_ratios": [1, 1.4]})
ax = axes[0]
ax.plot(b.date, b.sh_close, color=BLUE, lw=1.6)
ax.set_title("上证指数")
mark_keys(ax, top=True)
ax = axes[1]
ax.plot(b.date, b.big_trin_ma10, color=VIOLET, lw=2, label="权重（市值前 300）")
ax.plot(b.date, b.small_trin_ma10, color=ORANGE, lw=2, label="中小盘（其余）")
trin_axis(ax)
ax.set_title("分组 TRIN 10 日均线（对数坐标）")
ax.legend(loc="upper left", ncol=2)
mark_keys(ax)
fmt_x(ax)
fig.tight_layout()
fig.savefig("charts/3_big_vs_small.png")
plt.show()

# ===== 图 4：中信期货（代客）净持仓 vs 前 20 会员合计
#
# 净持仓 = 多单 − 空单（手）。数据是中金所公布的会员排名，"代客"即客户持仓合计。

if not fut.empty:
    fut["net"] = fut.long_hld - fut.short_hld
    fut["date"] = pd.to_datetime(fut.trade_date)
    citic = fut[fut.broker.astype(str).str.contains("中信期货")].groupby(["date", "product"]).net.sum().unstack()
    total = fut.groupby(["date", "product"]).net.sum().unstack()
    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
    ax = axes[0]
    ax.plot(b.date, b.hs300_close, color=BLUE, lw=1.6)
    ax.set_title("沪深300")
    colors = {"IF": BLUE, "IH": ORANGE, "IC": AQUA, "IM": VIOLET}
    for ax, df, title in [(axes[1], citic, "中信期货（代客）净持仓（手）"), (axes[2], total, "前 20 会员净持仓合计（手）")]:
        for p in ["IF", "IH", "IC", "IM"]:
            if p in df:
                ax.plot(df.index, df[p], color=colors[p], lw=1.6, label=p)
        ax.axhline(0, color=MUTED, lw=0.8)
        ax.set_title(title)
        ax.legend(loc="lower left", ncol=4)
    fmt_x(axes[2])
    fig.tight_layout()
    fig.savefig("charts/4_cffex.png")
    plt.show()

# ===== 图 5：规则检验——按 TRIN 区间分组，之后上证指数的平均涨跌幅
#
# 博主规则：单日 TRIN < 0.7 偏多、0.9 左右中性。Arms 原始规则：10 日均线 < 0.8 超买、> 1.2 超卖。
# 这里只看历史统计，样本量小、区间重叠，结论仅供参考。

for h in (5, 20):
    b[f"fwd{h}"] = (b.sh_close.shift(-h) / b.sh_close - 1) * 100
bins_d = [0, 0.5, 0.7, 1.0, 1.2, 2.0, np.inf]
lab_d = ["<0.5", "0.5–0.7", "0.7–1.0", "1.0–1.2", "1.2–2.0", ">2.0"]
bins_m = [0, 0.8, 1.0, 1.2, np.inf]
lab_m = ["<0.8", "0.8–1.0", "1.0–1.2", ">1.2"]
rows = []
for col, bins, labs, name in [("trin", bins_d, lab_d, "单日 TRIN"), ("trin_ma10", bins_m, lab_m, "TRIN 10 日均线")]:
    g = b.assign(bucket=pd.cut(b[col], bins, labels=labs)).groupby("bucket", observed=False)
    for lab, d in g:
        rows.append({"口径": name, "区间": lab, "天数": int(d.fwd20.notna().sum()),
                     "5日后平均涨跌%": d.fwd5.mean(), "20日后平均涨跌%": d.fwd20.mean(),
                     "20日后上涨概率%": (d.fwd20 > 0).mean() * 100 if d.fwd20.notna().any() else np.nan})
bt = pd.DataFrame(rows)
base = {"5日后平均涨跌%": b.fwd5.mean(), "20日后平均涨跌%": b.fwd20.mean(), "20日后上涨概率%": (b.fwd20 > 0)[b.fwd20.notna()].mean() * 100}
bt.round(2).to_csv("data/trin_backtest.csv", index=False, encoding="utf-8-sig")
print(bt.round(2).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1.5, 1]})
for ax, name in zip(axes, ["单日 TRIN", "TRIN 10 日均线"]):
    d = bt[bt["口径"] == name].reset_index(drop=True)
    x = np.arange(len(d))
    for i, (col, c) in enumerate([("5日后平均涨跌%", MUTED), ("20日后平均涨跌%", BLUE)]):
        bars = ax.bar(x + (i - 0.5) * 0.38, d[col], 0.36, color=c, label=col.replace("平均涨跌%", "平均涨跌"))
        ax.bar_label(bars, fmt="%.1f", fontsize=8, color=INK2, padding=2)
    ax.axhline(base["20日后平均涨跌%"], color=ORANGE, lw=1.2, ls="--", label="全样本 20 日平均")
    ax.axhline(0, color=MUTED, lw=0.8)
    ax.set_xticks(x, [f"{r}\n{n} 天" for r, n in zip(d["区间"], d["天数"])])
    ax.set_title(f"按 {name} 分组：之后上证平均涨跌（%）")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, fontsize=8)
    ax.grid(axis="x", visible=False)
fig.tight_layout()
fig.savefig("charts/5_backtest.png")
plt.show()


## 信号、回测与核对

- **信号回测**：每个信号之后 5 / 20 个交易日的上证涨跌，和任意一天买入的基准比较；同一波行情只算一次（独立次数），并单独剔除 2024 年 9–10 月行情做稳健性检验。独立次数少于 10 的不下结论。
- **大资金叙事核对**：权重（市值前 300）和中小盘分别算成交额口径 TRIN，用来核对"权重出货""蓝筹爆量"这类说法。
- **博主预测跟踪**： 里记录观点、基准日、方向和检验天数，到期自动判定；可以手动追加新观点。

In [ ]:
# ===== 信号、回测、大资金核对、博主预测跟踪 =====
# 依赖上面出图格里的配色和 b（行情 + 广度数据）
if "mark_keys" not in globals():
    raise RuntimeError("请先运行上面的出图格，再运行这一格")
try:
    from IPython.display import display
except ImportError:
    display = print

s = b.copy()
s["tot"] = s.n_up + s.n_dn
s["adr"] = s.n_up / s.tot                                     # 上涨/(上涨+下跌)
s["rana"] = (s.n_up - s.n_dn) / s.tot * 1000                  # 比例调整净上涨
s["mco"] = s.rana.ewm(alpha=0.10, adjust=False).mean() - s.rana.ewm(alpha=0.05, adjust=False).mean()  # 麦克莱伦：19日EMA−39日EMA
s["zema"] = s.adr.ewm(span=10, adjust=False).mean()            # Zweig：10日EMA
s["amt_rel"] = s.amt / s.amt.shift(1).rolling(60).median()     # 成交额 ÷ 此前60日中位数
for h in (5, 20):
    s[f"f{h}"] = (s.sh_close.shift(-h) / s.sh_close - 1) * 100

# ---- 信号定义（阈值在看结果之前按文献/通用约定定好）----
zweig = (s.zema > 0.615) & (s.zema.shift(1) <= 0.615) & (s.zema.rolling(11).min() < 0.40)
mco_thrust = (s.mco > 50) & (s.mco.shift(1) <= 50) & (s.mco.rolling(11).min() < -50)
SIGNALS = {
    "恐慌日：上涨家数<10%": s.breadth_up < 0.10,
    "普涨日：上涨家数>80%": s.breadth_up > 0.80,
    "Zweig 推力（10日EMA 0.40→0.615）": zweig,
    "麦克莱伦推力（−50→+50）": mco_thrust,
    "麦克莱伦 < −100（强抛压）": s.mco < -100,
    "TRIN 10日均 > 1.2（Arms 超卖）": s.trin_ma10 > 1.2,
    "TRIN 10日均 < 0.8（Arms 超买）": s.trin_ma10 < 0.8,
    "单日 TRIN > 2（极端）": s.trin > 2,
    "单日 TRIN < 0.5（极端）": s.trin < 0.5,
    "博主：成交额TRIN<0.7 多头进攻": s.trin_amt < 0.7,
    "博主：广度>75% 且成交额TRIN>1.5（9/21型）": (s.breadth_up > 0.75) & (s.trin_amt > 1.5),
    "博主：上证收涨 但广度<50%（9/22型）": (s.sh_pct > 0) & (s.breadth_up < 0.5),
}
RALLY = s.date.between("2024-09-10", "2024-11-15")   # 2024年9·24行情，单独剔除做稳健性检验


def decluster(mask, gap=20):
    """同一波行情只算一次：两次信号间隔不足 gap 个交易日的，只保留第一次。"""
    idx, last = [], -10**9
    for i in s.index[mask.fillna(False)]:
        if i - last >= gap:
            idx.append(i)
            last = i
    return idx


def stats(mask, sub=None):
    m = mask.fillna(False) & s.f20.notna()
    if sub is not None:
        m &= sub
    ind = [i for i in decluster(m)]
    d, dd = s[m], s.loc[ind]
    return {"天数": int(m.sum()), "独立次数": len(ind),
            "5日后平均%": d.f5.mean(), "20日后平均%": d.f20.mean(), "20日后上涨概率%": (d.f20 > 0).mean() * 100 if len(d) else np.nan,
            "独立·20日平均%": dd.f20.mean(), "独立·上涨概率%": (dd.f20 > 0).mean() * 100 if len(dd) else np.nan}


rows = []
for name, m in SIGNALS.items():
    r = {"信号": name, **stats(m)}
    ex = stats(m, ~RALLY)
    r["剔除24年9-10月·独立20日%"] = ex["独立·20日平均%"]
    r["剔除后独立次数"] = ex["独立次数"]
    last_i = s.index[m.fillna(False)]
    r["最近触发"] = s.trade_date[last_i[-1]] if len(last_i) else ""
    rows.append(r)
sig = pd.DataFrame(rows)
valid = s.f20.notna()
BASE = {"5日": s.f5[valid].mean(), "20日": s.f20[valid].mean(), "上涨概率": (s.f20[valid] > 0).mean() * 100,
        "20日_剔除": s.f20[valid & ~RALLY].mean()}
sig.round(2).to_csv("data/signals_backtest.csv", index=False, encoding="utf-8-sig")
print(f"基准（任意一天买入）：5日 {BASE['5日']:.2f}%，20日 {BASE['20日']:.2f}%，20日上涨概率 {BASE['上涨概率']:.1f}%；剔除24年9-10月后 20日 {BASE['20日_剔除']:.2f}%")
print("独立次数 < 10 的信号不下结论。")
display(sig.round(2))

# ---- 最近 10 个交易日：哪些信号亮了 ----
recent_idx = s.index[-10:]
lit = []
for name, m in SIGNALS.items():
    on = [s.trade_date[i] for i in recent_idx if bool(m.fillna(False)[i])]
    if on:
        lit.append({"信号": name, "触发日期": "、".join(on)})
lit = pd.DataFrame(lit, columns=["信号", "触发日期"])
print("\n最近 10 个交易日触发的信号：")
display(lit if not lit.empty else "无")


# ---- 大资金叙事核对：权重 vs 中小盘（成交额口径，与博主一致）----
def lean(x):
    if pd.isna(x):
        return ""
    return "资金偏向上涨股" if x < 0.8 else ("资金偏向下跌股" if x > 1.25 else "中性")


def vol_word(x):
    if pd.isna(x):
        return ""
    return "放量" if x > 1.3 else ("缩量" if x < 0.8 else "正常")


chk = s.tail(10).iloc[::-1]
check = pd.DataFrame({
    "日期": chk.trade_date, "上证涨跌%": chk.sh_pct.round(2), "上涨家数%": (chk.breadth_up * 100).round(1),
    "成交额/60日中位": chk.amt_rel.round(2), "量能": chk.amt_rel.map(vol_word),
    "权重 TRIN(额)": chk.big_trin_amt.round(2), "权重": chk.big_trin_amt.map(lean),
    "中小盘 TRIN(额)": chk.small_trin_amt.round(2), "中小盘": chk.small_trin_amt.map(lean),
    "TRIN(量)": chk.trin.round(2), "TRIN(额)": chk.trin_amt.round(2),
})
check.to_csv("data/daily_check.csv", index=False, encoding="utf-8-sig")
print("\n大资金叙事核对（最近 10 日）：TRIN<0.8 表示成交额集中在上涨股，>1.25 表示集中在下跌股")
display(check)

# ---- 博主预测跟踪：data/forecasts.csv 可手动追加 ----
FC = "data/forecasts.csv"
if not os.path.exists(FC):
    pd.DataFrame([
        {"发布日期": "20241007", "观点": "中期约3个月震荡下行", "基准日": "20241008", "方向": "跌", "检验交易日数": "20;40;60"},
        {"发布日期": "20260920", "观点": "9/22 短期风险，是高点", "基准日": "20260922", "方向": "跌", "检验交易日数": "5;10;20"},
        {"发布日期": "20261025", "观点": "10/23 只是反弹不是反转，半月到两月后回看仍在下跌趋势中", "基准日": "20261023", "方向": "跌", "检验交易日数": "10;20;40"},
    ]).to_csv(FC, index=False, encoding="utf-8-sig")
fc = pd.read_csv(FC, dtype=str)
pos = {d: i for i, d in enumerate(s.trade_date)}
ev = []
for _, r in fc.iterrows():
    i0 = pos.get(r["基准日"])
    for h in [int(x) for x in str(r["检验交易日数"]).split(";") if x.strip()]:
        row = {"观点": r["观点"], "基准日": r["基准日"], "方向": r["方向"], "检验": f"{h}个交易日后"}
        if i0 is None:
            row.update({"到期日": "", "上证涨跌%": np.nan, "判定": "基准日不在数据内"})
        elif i0 + h >= len(s):
            row.update({"到期日": f"还差 {i0 + h - len(s) + 1} 个交易日", "上证涨跌%": np.nan, "判定": "待验证"})
        else:
            ret = (s.sh_close.iloc[i0 + h] / s.sh_close.iloc[i0] - 1) * 100
            ok = ret < 0 if r["方向"] == "跌" else ret > 0
            row.update({"到期日": s.trade_date.iloc[i0 + h], "上证涨跌%": round(ret, 2), "判定": "对" if ok else "错"})
        ev.append(row)
fce = pd.DataFrame(ev)
fce.to_csv("data/forecasts_eval.csv", index=False, encoding="utf-8-sig")
print("\n博主预测跟踪（以上证收盘价判定；在 data/forecasts.csv 里可追加新观点）：")
display(fce)

# ---- 图 6：信号时间线 ----
fig, axes = plt.subplots(3, 1, figsize=(12, 9.5), sharex=True, gridspec_kw={"height_ratios": [1.4, 1, 1]})
ax = axes[0]
ax.plot(s.date, s.sh_close, color=BLUE, lw=1.4, zorder=1)
for name, m, c, mk in [("恐慌日：上涨家数<10%", SIGNALS["恐慌日：上涨家数<10%"], ORANGE, "v"),
                       ("普涨日：上涨家数>80%", SIGNALS["普涨日：上涨家数>80%"], AQUA, "^"),
                       ("Zweig 推力", zweig, VIOLET, "*"),
                       ("TRIN 10日均>1.2", SIGNALS["TRIN 10日均 > 1.2（Arms 超卖）"], MAGENTA, "D")]:
    mm = m.fillna(False)
    ax.scatter(s.date[mm], s.sh_close[mm], color=c, marker=mk, s=60 if mk == "*" else 30,
               edgecolor="#fcfcfb", linewidth=0.8, zorder=3, label=name)
ax.set_title("上证指数与信号")
ax.legend(loc="upper left", ncol=2, fontsize=8)
mark_keys(ax, top=True)
ax = axes[1]
ax.plot(s.date, s.zema, color=VIOLET, lw=1.6)
ax.axhline(0.40, color=GREEN, lw=0.8, ls="--")
ax.axhline(0.615, color=RED, lw=0.8, ls="--")
ax.set_ylim(0, 1)
ax.set_title("Zweig：上涨占比 10日EMA（0.40 → 0.615 为推力）")
mark_keys(ax)
ax = axes[2]
ax.fill_between(s.date, 0, s.mco, where=s.mco >= 0, color=AQUA, alpha=0.5, linewidth=0)
ax.fill_between(s.date, 0, s.mco, where=s.mco < 0, color=ORANGE, alpha=0.5, linewidth=0)
for y in (50, -50):
    ax.axhline(y, color=MUTED, lw=0.8, ls="--")
ax.axhline(-100, color=RED, lw=0.8, ls="--")
ax.axhline(0, color=MUTED, lw=0.8)
ax.set_title("麦克莱伦振荡指标（比例调整，19日EMA − 39日EMA）")
mark_keys(ax)
fmt_x(ax)
fig.tight_layout()
fig.savefig("charts/6_signals.png")
plt.show()

# ---- 图 7：各信号之后 20 日表现 vs 基准 ----
sp = sig.iloc[::-1].reset_index(drop=True)
y = np.arange(len(sp))
fig, ax = plt.subplots(figsize=(12, 0.48 * len(sp) + 1.6))
b1 = ax.barh(y + 0.2, sp["20日后平均%"], 0.38, color=MUTED, label="全部触发日")
b2 = ax.barh(y - 0.2, sp["独立·20日平均%"], 0.38, color=BLUE, label="独立信号（同一波只算一次）")
for bars in (b1, b2):
    ax.bar_label(bars, fmt="%.1f", fontsize=8, color=INK2, padding=3)
ax.axvline(BASE["20日"], color=ORANGE, lw=1.2, ls="--", label=f"基准 {BASE['20日']:.2f}%")
ax.axvline(0, color=MUTED, lw=0.8)
ax.set_yticks(y, [f"{n}（{k} 次{"，样本少" if k < 10 else ""}）" for n, k in zip(sp["信号"], sp["独立次数"])])
ax.set_title("各信号之后 20 个交易日上证平均涨跌（%）")
ax.legend(loc="lower right", fontsize=8)
ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig("charts/7_signal_backtest.png")
plt.show()


## 写 README（仓库首页显示）

In [ ]:
def pct(v):
    return "" if pd.isna(v) else f"{v * 100:.1f}%"


def num(v, f="{:.2f}"):
    return "" if pd.isna(v) else f.format(v)


def md_table(df):
    head = "| " + " | ".join(df.columns) + " |\n|" + "---|" * len(df.columns) + "\n"
    return head + "".join("| " + " | ".join(str(v) for v in r) + " |\n" for r in df.values)


kt = pd.DataFrame({
    "日期": key.label, "上涨家数占比": key.breadth_up.map(pct), "涨停": key.get("limit_up", pd.Series(dtype=float)).map(lambda v: num(v, "{:.0f}")),
    "TRIN": key.trin.map(num), "TRIN(成交额)": key.trin_amt.map(num), "TRIN 10日均": key.trin_ma10.map(num),
    "权重 TRIN": key.big_trin.map(num), "中小盘 TRIN": key.small_trin.map(num), "成交额(亿)": key.amt.map(lambda v: num(v, "{:.0f}")),
})
recent = b.tail(10)
rt = pd.DataFrame({
    "日期": recent.trade_date, "上证": recent.sh_close.map(lambda v: num(v)), "上涨家数占比": recent.breadth_up.map(pct),
    "TRIN": recent.trin.map(num), "TRIN 10日均": recent.trin_ma10.map(num),
    "权重 TRIN": recent.big_trin.map(num), "中小盘 TRIN": recent.small_trin.map(num),
    "成交额(亿)": recent.amt.map(lambda v: num(v, "{:.0f}")),
})
lit_md = md_table(lit) if not lit.empty else "无\n"
sig_md = md_table(sig.assign(**{c: sig[c].map(lambda v: num(v, "{:.2f}")) for c in sig.columns if sig[c].dtype.kind == "f"})[
    ["信号", "独立次数", "20日后平均%", "独立·20日平均%", "独立·上涨概率%", "剔除24年9-10月·独立20日%", "最近触发"]])
fc_md = md_table(fce.fillna(""))
chk_md = md_table(check)
readme = f"""# A 股市场广度 / TRIN 跟踪

自动更新：每个交易日收盘后由 GitHub Actions 运行。最新数据日期：**{last.trade_date}**（共 {len(b)} 个交易日，起始 {b.trade_date.iloc[0]}）。

**TRIN（Arms Index）=（上涨家数 ÷ 下跌家数）÷（上涨成交量 ÷ 下跌成交量）**，平盘不计。小于 1 表示成交量相对集中在上涨股。
Arms 原始用法看 10 日均线：低于 0.8 视为超买，高于 1.2 视为超卖。范围：沪深 A 股（不含北交所）。

## 最近 10 个交易日

{md_table(rt.iloc[::-1])}
## 博主提到的几天

{md_table(kt) if not kt.empty else "数据区间内暂无这些日期。"}
## 最近 10 个交易日触发的信号

{lit_md}
## 博主预测跟踪

以上证收盘价判定，观点记录在 [`data/forecasts.csv`](data/forecasts.csv)。

{fc_md}
## 大资金叙事核对（最近 10 日）

成交额口径 TRIN：小于 0.8 表示成交额集中在上涨股，大于 1.25 表示集中在下跌股。权重 = 当日市值前 300。

{chk_md}
## 信号回测

基准（任意一天买入）20 日平均 {BASE["20日"]:.2f}%。独立次数少于 10 的不下结论。

{sig_md}
## 图表

![总览](charts/1_overview.png)

![关键日期](charts/2_key_days.png)

![权重 vs 中小盘](charts/3_big_vs_small.png)

![中金所持仓](charts/4_cffex.png)

![规则检验](charts/5_backtest.png)

![信号时间线](charts/6_signals.png)

![信号回测](charts/7_signal_backtest.png)

明细：[`data/trin_backtest.csv`](data/trin_backtest.csv)、[`data/signals_backtest.csv`](data/signals_backtest.csv)、[`data/daily_check.csv`](data/daily_check.csv)、[`data/forecasts_eval.csv`](data/forecasts_eval.csv)。

## 使用

1. 仓库 **Settings → Secrets and variables → Actions → New repository secret**，名称 `TUSHARE_TOKEN`，值填 tushare token。
2. **Actions → 更新 TRIN 数据 → Run workflow** 手动运行一次（首次从 2024 年拉起，约 30–60 分钟）。之后每个交易日北京时间 18:00 左右自动运行，只拉新增日期。
3. 运行完成后刷新本页即可看到最新结果；执行过的 notebook 也会提交回仓库。

文件：`pull_data.ipynb` 拉数据 → `data/*.csv`；`charts.ipynb` 出图 → `charts/*.png` 并重写本 README。
"""
open("README.md", "w", encoding="utf-8").write(readme)
print("README.md 已更新")
